# Zero-Shot Prompting — Ollama (gemma4:31b)
Local inference via Ollama. No API key required. GPU-accelerated on your RTX A6000.


In [1]:
# Install requests if needed (ollama uses local REST, no special SDK required)
# !pip install requests pillow
import requests, json, base64, os, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
print('Imports OK')
# Quick health-check — make sure Ollama is running
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama is running. Installed models: {models}')
except Exception as e:
    print(f'ERROR: Ollama not reachable — is it running? ({e})')
    print('Start it with: ollama serve')


Imports OK
Ollama is running. Installed models: ['gemma4:31b']


# NeurIPS Computational Resources & Reproducibility Metadata
Required by **NeurIPS 2025 Paper Checklist §8**. Run to print a live hardware/version summary.


In [2]:
"""
NeurIPS 2025 Checklist S8 - Computational Resources (Ollama / Local)
=====================================================================
Hardware
  GPU      : NVIDIA RTX A6000 (48 GB VRAM)
  CPU      : [TODO: e.g. AMD EPYC 7542 32-core]
  RAM      : [TODO: e.g. 256 GB DDR4]
  OS       : Windows 11 / Ubuntu 22.04
  Provider : Local on-premise workstation (Ollama)

Model & Inference
  Model      : gemma4:31b  (Google DeepMind Gemma 4, 31B dense, Q4_K_M via Ollama)
  num_predict: 4096 per call
  Temp       : 0.0 (Zero-Shot)

API Calls per Video  (C = ceil(frames / BATCH_SIZE))
  Zero-Shot  : C + 1 (batches + synthesis)
  Total/video (50 frames, C=3 at BATCH_SIZE=20) ~ 4 calls

Reproducibility
  - Checkpoint files save progress after every video (atomic write)
  - Re-running any cell after failure resumes with zero extra cost
  - All raw Ollama responses saved as JSON before post-processing
  - No internet required after model pull
"""
import subprocess, platform, datetime
print('=' * 64)
print(f'  Ollama Compute Summary  - {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('=' * 64)
print(f'  Python  : {platform.python_version()}')
print(f'  OS      : {platform.platform()}')
try:
    smi = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader'],
        text=True).strip()
    for line in smi.split('\n'):
        print(f'  GPU     : {line.strip()}')
except Exception:
    print('  GPU     : nvidia-smi not available')
try:
    r = requests.get('http://localhost:11434/api/version', timeout=5)
    print(f'  Ollama  : v{r.json().get("version", "unknown")}')
except Exception:
    print('  Ollama  : version check failed')
print(f'  Model   : gemma4:31b')
print(f'  Chunks  : {20} frames/batch  |  num_predict : 4096')
print(f'  Retry   : 5 attempts, exponential back-off')
print('=' * 64)


  Ollama Compute Summary  - 2026-05-01 13:52
  Python  : 3.10.11
  OS      : Windows-10-10.0.26100-SP0
  GPU     : NVIDIA H100 NVL, 95830 MiB, 1 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB, 1 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB, 1 MiB
  GPU     : NVIDIA H100 NVL, 95830 MiB, 1 MiB
  Ollama  : v0.22.1
  Model   : gemma4:31b
  Chunks  : 20 frames/batch  |  num_predict : 4096
  Retry   : 5 attempts, exponential back-off


# ZERO-SHOT
* Clear, direct instructions
* No examples or demonstrations
* Each task stands alone
* Direct analysis without prior context


In [ ]:
import os, json, base64, requests, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================================================================
#  CONFIGURATION
# ================================================================
OLLAMA_URL     = 'http://localhost:11434/api/chat'
MODEL_NAME     = 'gemma4:31b'
FRAMES_DIR     = r'C:\Opeyemi\PROMPTS\FRAMES'
RESULTS_BASE   = r'C:\Opeyemi\PROMPTS\RESULTS'
FRAME_EXT      = '.jpg'
FRAME_INTERVAL = 1
BATCH_SIZE     = 20
MAX_WORKERS    = 4
SAVE_DIR       = r'C:\Opeyemi\PROMPTS\RESULTS\OLLAMA\ZERO'
os.makedirs(SAVE_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(SAVE_DIR, 'zero_shot_ollama_checkpoint.json')

# ================================================================
#  FRAME HELPERS
# ================================================================
def extract_frame_number(filename):
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r'frame[_\-]?(\d+)', name, _re.IGNORECASE)
    if m: return int(m.group(1))
    nums = _re.findall(r'\d+', name)
    return int(nums[-1]) if nums else 0

def discover_all_videos_and_frames(frames_dir=None):
    if frames_dir is None: frames_dir = FRAMES_DIR
    print(f'\n=== DISCOVERING FRAMES ===')
    print(f'    Root : {frames_dir}')
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f'  ERROR: FRAMES_DIR not found: {frames_dir}'); return all_videos
    crime_types = sorted([d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith('_')])
    print(f'  Categories : {crime_types}')
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))])
        print(f'    {crime_type:20s}: {len(video_stems)} videos')
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number)
            if not frame_files:
                print(f'      WARNING: no {FRAME_EXT} frames in {vdir} - skipping'); continue
            key = f'{crime_type}_{video_stem}'
            all_videos[key] = {'crime_type': crime_type, 'video_id': video_stem,
                               'frames_dir': vdir, 'frames': frame_files}
    print(f'  Total videos ready: {len(all_videos)}')
    return all_videos

def load_frames_for_video(video_info, frame_interval=1):
    frames_data = {}
    vdir = video_info['frames_dir']
    frame_files = video_info['frames']
    video_id = video_info['video_id']
    selected = frame_files[::frame_interval]
    label = 'ALL' if frame_interval == 1 else f'every {frame_interval}th'
    print(f'  Loading {len(selected)} frames ({label}) for {video_id} ...')
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, 'rb') as fh:
                frames_data[ff] = base64.b64encode(fh.read()).decode('utf-8')
        except Exception as e:
            print(f'    ERROR loading {ff}: {e}')
    print(f'  Loaded {len(frames_data)}/{len(selected)} frames OK')
    return frames_data

# ================================================================
#  CHECKPOINT
# ================================================================
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as f: data = json.load(f)
            print(f'  Checkpoint: {len(data.get("completed_videos", []))} videos done - skipping them.')
            return data
        except Exception as e:
            print(f'  Could not read checkpoint ({e}) - starting fresh.')
    return {'completed_videos': [], 'results': {}}

def save_checkpoint(data):
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + '.tmp'
    with open(tmp, 'w') as f: json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)

# ================================================================
#  OLLAMA REQUEST HELPER
# ================================================================
def make_ollama_request_robust(messages, temperature=0.0, max_retries=5, base_wait=5):
    """
    Fault-tolerant Ollama API call.
    Retries on network errors and server errors.
    Returns response text on success, or error string on failure.
    """
    import random
    payload = {
        'model': MODEL_NAME,
        'messages': messages,
        'stream': False,
        'options': {
            'temperature': temperature,
            'num_predict': 4096,
            'num_gpu': 99,          # offload ALL layers to GPU (99 = use all available)
            'num_thread': 8,        # CPU threads for non-GPU ops (preprocessing etc.)
            'num_batch': 512,       # prompt eval batch size — bigger = faster on 48GB VRAM
            'num_ctx': 8192,        # context window — 8K is plenty per call, saves VRAM
            'low_vram': False,      # we have 48GB — never throttle
            'f16_kv': True,         # use float16 for KV cache — faster, still accurate
            'use_mmap': True,       # memory-map model weights — faster cold start
            'use_mlock': False,     # don't lock RAM (not needed, full GPU offload)
        }
    }
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=300)
            if response.status_code == 200:
                data = response.json()
                return data.get('message', {}).get('content', str(data))
            if response.status_code >= 500:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
                wait = min(wait, 120)
                print(f'[Retry {attempt}/{max_retries}] HTTP {response.status_code}. Waiting {wait:.1f}s ...')
                time.sleep(wait); continue
            return f'ERROR: HTTP {response.status_code}: {response.text[:200]}'
        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
            wait = min(wait, 120)
            print(f'[Retry {attempt}/{max_retries}] Network error: {e}. Waiting {wait:.1f}s ...')
            time.sleep(wait)
        except Exception as e:
            if attempt < max_retries:
                time.sleep(base_wait * attempt)
            else:
                return f'ERROR: {e}'
    return f'ERROR: All {max_retries} attempts exhausted'

def frames_to_ollama_content(frames_data, frame_names, prompt_text):
    """Build Ollama multimodal message content: images + text."""
    # Ollama accepts images as base64 strings in the 'images' field
    images = [frames_data[fname] for fname in frame_names if fname in frames_data]
    return {'role': 'user', 'content': prompt_text, 'images': images}

# ================================================================
#  ZERO-SHOT ANALYZER
# ================================================================
class ZeroShotOllamaAnalyzer:
    """
    Zero-shot crime analysis using gemma4:31b via Ollama.
    Sends frames in batches of BATCH_SIZE, collects per-batch observations,
    then synthesizes into a final structured report.
    """
    def analyze_frames(self, frames_data, video_id, crime_type):
        print(f'\n  [Zero-Shot] {video_id} | {len(frames_data)} frames ...')
        frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
        batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
        batch_summaries = []

        for idx, batch in enumerate(batches, 1):
            print(f'    Batch {idx}/{len(batches)} ({len(batch)} frames) ...')
            prompt = (
                f'Analyze this batch of security camera frames (Batch {idx} of {len(batches)}).\n'
                'Describe: (1) What is happening, (2) Suspicious behaviour if any, '
                '(3) Probable crime type from: Abuse, Arrest, Arson, Assault, Burglary, '
                'Explosion, Fighting, RoadAccidents, Robbery, Shooting, Shoplifting, '
                'Stealing, Vandalism, or Normal. (4) Confidence 0-100%.'
            )
            msg = frames_to_ollama_content(frames_data, batch, prompt)
            summary = make_ollama_request_robust([msg], temperature=0.0)
            batch_summaries.append(summary)
            print(f'      Response: {len(summary)} chars')

        # Synthesis — text only, no images
        print('    Synthesizing batches ...')
        formatted = '\n\n'.join(f'--- Batch {i+1} ---\n{s}' for i, s in enumerate(batch_summaries))
        synth_prompt = (
            f'You analyzed {len(batches)} batches of {len(frames_data)} security camera frames.\n\n'
            f'Per-batch findings:\n{formatted}\n\n'
            'Provide the FINAL STRUCTURED REPORT:\n'
            'PRIMARY CLASSIFICATION: [crime type]\n'
            'CONFIDENCE LEVEL: [0-100%]\n'
            'SEVERITY: [Low/Medium/High/Critical]\n'
            'KEY EVIDENCE:\n- [point 1]\n- [point 2]\n- [point 3]\n'
            'ALTERNATIVE INTERPRETATIONS: [other explanations]\n'
            'RECOMMENDED LAW ENFORCEMENT RESPONSE: [actions]'
        )
        final = make_ollama_request_robust(
            [{'role': 'user', 'content': synth_prompt}], temperature=0.0)

        return {
            'video_id': video_id, 'crime_type': crime_type,
            'frames_analyzed': len(frames_data), 'total_batches': len(batches),
            'batch_size': BATCH_SIZE, 'prompting_technique': 'ZERO-SHOT',
            'model': MODEL_NAME, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'batch_summaries': batch_summaries, 'final_analysis': final
        }

# ================================================================
#  PARALLEL VIDEO PROCESSING
# ================================================================
def process_all_crime_folders():
    analyzer   = ZeroShotOllamaAnalyzer()
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print('No videos found! Verify FRAMES_DIR path.'); return {}
    cp          = load_checkpoint()
    all_results = cp.get('results', {})
    done_set    = set(cp.get('completed_videos', []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f'\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}')

    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        with print_lock: print(f'\n  [START] {vkey}')
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames: return vkey, None, 'no frames'
            res = analyzer.analyze_frames(frames, vinfo['video_id'], vinfo['crime_type'])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({'completed_videos': list(done_set), 'results': all_results})
            with print_lock: print(f'  [DONE]  {vkey}  ({len(done_set)}/{total})')
            return vkey, res, None
        except Exception as e:
            with print_lock: print(f'  [ERROR] {vkey}: {e}')
            with checkpoint_lock:
                save_checkpoint({'completed_videos': list(done_set), 'results': all_results})
            return vkey, None, f'error: {e}'

    print(f'\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...')
    print('  NOTE: Ollama serves requests sequentially; threads queue automatically.')
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err: skipped.append(f'{vkey} ({err})')

    ts = time.strftime('%Y%m%d_%H%M%S')
    summary = os.path.join(SAVE_DIR, f'zero_shot_ollama_summary_{ts}.json')
    with open(summary, 'w') as f: json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f'skipped_{ts}.txt'), 'w') as f:
            f.write('\n'.join(skipped))
    print(f'\nDone. Summary -> {summary}')
    print(f'  Processed: {len(all_results)} | Skipped: {len(skipped)}')
    return all_results

# ================================================================
#  RUN
# ================================================================
def run():
    print('Zero-Shot Crime Video Analysis — Ollama gemma4:31b')
    print('=' * 55)
    print('Testing directory access...')
    for path in [FRAMES_DIR, SAVE_DIR]:
        exists = os.path.exists(path)
        print(f'  {path} -> exists={exists}')
        if exists:
            try: print(f'    Contains {len(os.listdir(path))} items')
            except Exception as e: print(f'    Error: {e}')
    if not os.path.exists(FRAMES_DIR):
        print(f'ERROR: Frames directory not found: {FRAMES_DIR}'); return
    # Verify model is available
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=5)
        models = [m['name'] for m in r.json().get('models', [])]
        if not any(MODEL_NAME.split(':')[0] in m for m in models):
            print(f'WARNING: {MODEL_NAME} may not be pulled yet.')
            print(f'Run: ollama pull {MODEL_NAME}')
        else:
            print(f'Model {MODEL_NAME} is available locally.')
    except Exception as e:
        print(f'WARNING: Could not check Ollama model list: {e}')
    results = process_all_crime_folders()
    print('\n' + '=' * 55)
    print('ZERO-SHOT (OLLAMA) COMPLETE!')
    print(f'Videos processed: {len(results)}')
    print('=' * 55)

if __name__ == '__main__':
    run()


Zero-Shot Crime Video Analysis — Ollama gemma4:31b
Testing directory access...
  C:\Opeyemi\PROMPTS\FRAMES -> exists=True
    Contains 13 items
  C:\Opeyemi\PROMPTS\RESULTS\OLLAMA\ZERO -> exists=True
    Contains 1 items
Model gemma4:31b is available locally.

=== DISCOVERING FRAMES ===
    Root : C:\Opeyemi\PROMPTS\FRAMES
  Categories : ['Abuse', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
    Abuse               : 50 videos
    Assault             : 12 videos
    Burglary            : 100 videos
    Explosion           : 50 videos
    Fighting            : 50 videos
    RoadAccidents       : 150 videos
    Robbery             : 150 videos
    Shooting            : 50 videos
    Shoplifting         : 50 videos
    Stealing            : 100 videos
    Vandalism           : 50 videos
  Total videos ready: 812
  Checkpoint: 1 videos done - skipping them.

Videos: total=812 | done=1 | remaining=811

  Laun

In [ ]:
run()
